# 🏭 Visualisation du Process Industriel Multi-Sites

Ce notebook permet de visualiser et d'analyser les flux de votre installation industrielle répartie sur plusieurs sites.  
Les données sont chargées depuis des fichiers JSON évolutifs.

---
**Fichiers de configuration :**
- `sites.json` — Définition des sites industriels
- `matieres_premieres.json` — Matières premières et stocks
- `unites_production.json` — Unités de production et flux

**Librairies :** `networkx`, `plotly`, `pandas`, `json`

## 1. 📦 Installation et imports

In [1]:
# Installation des dépendances (décommenter si nécessaire)
# !pip install networkx plotly pandas

import json
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports OK')

✅ Imports OK


## 2. 📂 Chargement des données

In [2]:
# ── Chargement des fichiers JSON ──────────────────────────────────────────────
BASE_DIR = Path('.')  # Adapter si les fichiers sont dans un sous-dossier

def load_json(filename):
    path = BASE_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Fichier introuvable : {path}')
    with open(path, encoding='utf-8') as f:
        return json.load(f)

data_sites  = load_json('sites.json')['sites']
data_mp     = load_json('matieres_premieres.json')['matieres_premieres']
data_up     = load_json('unites_production.json')['unites_production']

# DataFrames
df_sites = pd.DataFrame(data_sites)
df_mp    = pd.DataFrame(data_mp)
df_up    = pd.DataFrame(data_up)

print(f'✅ {len(df_sites)} sites  |  {len(df_mp)} matières premières  |  {len(df_up)} unités de production')

✅ 4 sites  |  5 matières premières  |  7 unités de production


## 3. 🔍 Exploration rapide des données

In [3]:
print('\n🏭 SITES')
display(df_sites[['code','nom','localisation','type','actif']])

print('\n🧪 MATIÈRES PREMIÈRES')
display(df_mp[['code','nom','volume_stock','unite_volume','site_code','type']])

print('\n⚙️ UNITÉS DE PRODUCTION')
display(df_up[['code','nom','produit','site_code','oee','cmj','capacite_max_j','type_ligne','type_produit']])


🏭 SITES


,code,nom,localisation,type,actif
0,SITE-A,Site Alpha,Lyon,production,True
1,SITE-B,Site Beta,Grenoble,production,True
2,SITE-C,Site Gamma,Marseille,finition,True
3,SITE-D,Site Delta,Paris,logistique,True



🧪 MATIÈRES PREMIÈRES


,code,nom,volume_stock,unite_volume,site_code,type
0,MP-001,Résine PET,50000,kg,SITE-A,matiere_premiere
1,MP-002,Colorant Bleu,2000,kg,SITE-A,additif
2,MP-003,Acier galvanisé,120000,kg,SITE-B,matiere_premiere
3,MP-004,Huile de coupe,8000,L,SITE-B,consommable
4,MP-005,Emballage carton,15000,unité,SITE-C,emballage



⚙️ UNITÉS DE PRODUCTION


,code,nom,produit,site_code,oee,cmj,capacite_max_j,type_ligne,type_produit
0,UP-001,Ligne Injection A1,Préforme PET,SITE-A,82.5,8000,10000,principale,produit_intermediaire
1,UP-002,Ligne Soufflage B1,Flacon PET 500mL,SITE-A,78.0,45000,60000,principale,produit_intermediaire
2,UP-003,Ligne Soufflage B2,Flacon PET 1L,SITE-A,75.0,28000,38000,secondaire,produit_intermediaire
3,UP-004,Ligne Emboutissage C1,Pièce acier stampée,SITE-B,88.0,6500,7500,principale,produit_intermediaire
4,UP-005,Ligne Remplissage D1,Produit conditionné,SITE-B,80.0,35000,45000,principale,produit_fini
5,UP-006,Ligne Assemblage E1,Sous-ensemble mécanique,SITE-B,85.5,5800,7000,principale,produit_fini
6,UP-007,Ligne Emballage F1,Produit fini emballé,SITE-C,91.0,38000,42000,principale,produit_fini


## 4. 🕸️ Construction du graphe de flux

In [4]:
# ── Couleurs par type de nœud ─────────────────────────────────────────────────
COLORS = {
    'matiere_premiere' : '#E74C3C',
    'additif'          : '#E74C3C',
    'consommable'      : '#E74C3C',
    'emballage'        : '#E74C3C',
    'produit_intermediaire': '#F39C12',
    'produit_fini'     : '#27AE60',
    'site'             : '#2C3E50',
}

# Index pratiques
site_color  = {s['code']: s['couleur'] for s in data_sites}
site_name   = {s['code']: s['nom'] for s in data_sites}
mp_index    = {m['code']: m for m in data_mp}
up_index    = {u['code']: u for u in data_up}

# ── Création du graphe dirigé ─────────────────────────────────────────────────
G = nx.DiGraph()

# Nœuds — Matières premières
for mp in data_mp:
    G.add_node(
        mp['code'],
        label=mp['nom'],
        type='matiere_premiere',
        subtype=mp['type'],
        site=mp['site_code'],
        volume=mp['volume_stock'],
        unite=mp['unite_volume'],
        color=COLORS.get(mp['type'], '#BDC3C7'),
    )

# Nœuds — Unités de production
for up in data_up:
    G.add_node(
        up['code'],
        label=up['nom'],
        type=up['type_produit'],
        site=up['site_code'],
        oee=up['oee'],
        cmj=up['cmj'],
        capacite_max=up['capacite_max_j'],
        type_ligne=up['type_ligne'],
        produit=up['produit'],
        color=COLORS.get(up['type_produit'], '#BDC3C7'),
    )

# Arêtes — MP → UP
for up in data_up:
    for mp_code in up.get('matieres_premieres_aval', []):
        if mp_code.startswith('MP-') and mp_code in G:
            G.add_edge(mp_code, up['code'], type='approvisionnement')

# Arêtes — UP → UP (flux inter-unités)
for up in data_up:
    for next_code in up.get('produits_suivants', []):
        if next_code in G:
            is_main = up['type_ligne'] == 'principale'
            G.add_edge(up['code'], next_code,
                       type='flux_principal' if is_main else 'flux_secondaire')

print(f'✅ Graphe : {G.number_of_nodes()} nœuds  |  {G.number_of_edges()} arêtes')

✅ Graphe : 12 nœuds  |  12 arêtes


## 5. 🗺️ Visualisation principale — Graphe de flux multi-sites

In [5]:
def build_plotly_graph(G, title='Flux Industriel Multi-Sites',
                        layout_algo='dot', show_mp=True):
    """
    Génère un graphe Plotly interactif du flux industriel.
    layout_algo : 'dot' (hiérarchique), 'spring', 'kamada_kawai'
    show_mp     : afficher ou masquer les nœuds matières premières
    """

    # Filtrage optionnel
    nodes = [n for n in G.nodes if show_mp or not n.startswith('MP-')]
    subG  = G.subgraph(nodes)

    # Positionnement
    if layout_algo == 'dot':
        try:
            pos = nx.nx_agraph.graphviz_layout(subG, prog='dot')
        except Exception:
            pos = nx.spring_layout(subG, seed=42, k=2)
    elif layout_algo == 'kamada_kawai':
        pos = nx.kamada_kawai_layout(subG)
    else:
        pos = nx.spring_layout(subG, seed=42, k=2)

    # ── Arêtes ───────────────────────────────────────────────────────────────
    edge_traces = []
    for u, v, data in subG.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_type = data.get('type', 'flux')
        color  = '#2C3E50' if edge_type == 'flux_principal' else \
                 '#95A5A6' if edge_type == 'flux_secondaire' else '#E74C3C'
        width  = 3 if edge_type == 'flux_principal' else 1.5
        dash   = 'solid' if 'principal' in edge_type or 'appro' in edge_type else 'dash'

        edge_traces.append(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None],
            mode='lines',
            line=dict(width=width, color=color, dash=dash),
            hoverinfo='none',
            showlegend=False,
        ))

        # Flèche (annotation)
    # ── Nœuds ────────────────────────────────────────────────────────────────
    node_x, node_y, node_text, node_hover = [], [], [], []
    node_colors, node_sizes, node_symbols = [], [], []

    for node in subG.nodes:
        attr = subG.nodes[node]
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(f"<b>{attr.get('label', node)}</b><br><i>{node}</i>")
        node_colors.append(attr.get('color', '#BDC3C7'))

        is_mp   = node.startswith('MP-')
        is_main = attr.get('type_ligne') == 'principale'
        node_sizes.append(18 if is_mp else (28 if is_main else 22))
        node_symbols.append('diamond' if is_mp else
                             'square' if attr.get('type') == 'produit_fini' else 'circle')

        # Tooltip
        site = site_name.get(attr.get('site',''), attr.get('site',''))
        if is_mp:
            mp = mp_index.get(node, {})
            hover = (f"<b>{attr.get('label','')}</b><br>"
                     f"Code : {node}<br>"
                     f"Site : {site}<br>"
                     f"Stock : {mp.get('volume_stock','-')} {mp.get('unite_volume','')}")
        else:
            up = up_index.get(node, {})
            oee_color = '🟢' if attr.get('oee',0)>=85 else '🟡' if attr.get('oee',0)>=75 else '🔴'
            hover = (f"<b>{attr.get('label','')}</b><br>"
                     f"Code : {node}<br>"
                     f"Site : {site}<br>"
                     f"Produit : {up.get('produit','-')}<br>"
                     f"OEE : {oee_color} {attr.get('oee','-')}%<br>"
                     f"CMJ : {attr.get('cmj','-')} {up.get('unite_capacite','')}")
        node_hover.append(hover)

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        hovertext=node_hover,
        hoverinfo='text',
        text=[subG.nodes[n].get('label', n) for n in subG.nodes],
        textposition='top center',
        textfont=dict(size=10, family='Arial'),
        marker=dict(
            size=node_sizes,
            color=node_colors,
            symbol=node_symbols,
            line=dict(width=2, color='white'),
        ),
        showlegend=False,
    )

    # ── Légende manuelle ─────────────────────────────────────────────────────
    legend_items = [
        go.Scatter(x=[None], y=[None], mode='markers',
                   marker=dict(size=12, color='#E74C3C', symbol='diamond'),
                   name='Matière première'),
        go.Scatter(x=[None], y=[None], mode='markers',
                   marker=dict(size=12, color='#F39C12', symbol='circle'),
                   name='Produit intermédiaire'),
        go.Scatter(x=[None], y=[None], mode='markers',
                   marker=dict(size=12, color='#27AE60', symbol='square'),
                   name='Produit fini'),
        go.Scatter(x=[None], y=[None], mode='lines',
                   line=dict(width=3, color='#2C3E50'), name='Flux principal'),
        go.Scatter(x=[None], y=[None], mode='lines',
                   line=dict(width=2, color='#95A5A6', dash='dash'), name='Flux secondaire'),
    ]

    fig = go.Figure(data=edge_traces + [node_trace] + legend_items)
    fig.update_layout(
        title=dict(text=title, font=dict(size=18), x=0.5),
        showlegend=True,
        legend=dict(orientation='v', x=1.01, y=1),
        hovermode='closest',
        margin=dict(b=20, l=5, r=150, t=50),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        paper_bgcolor='#F8F9FA',
        plot_bgcolor='#F8F9FA',
        height=650,
    )
    return fig


fig = build_plotly_graph(G, title='🏭 Flux Industriel — Vue complète', show_mp=True)
fig.show()

## 6. 🔗 Vue flux principaux uniquement (sans matières premières)

In [6]:
fig2 = build_plotly_graph(
    G,
    title='⚙️ Flux Industriel — Lignes principales',
    show_mp=False
)
fig2.show()

## 7. 📊 Dashboard OEE & Capacités par unité

In [7]:
fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('OEE par unité de production (%)', 'CMJ vs Capacité max (unités/j)'),
)

# Tri par OEE
df_sorted = df_up.sort_values('oee')

oee_colors = df_sorted['oee'].apply(
    lambda v: '#27AE60' if v >= 85 else '#F39C12' if v >= 75 else '#E74C3C'
)

# OEE
fig3.add_trace(go.Bar(
    y=df_sorted['nom'],
    x=df_sorted['oee'],
    orientation='h',
    marker_color=oee_colors,
    text=df_sorted['oee'].astype(str) + '%',
    textposition='outside',
    name='OEE',
), row=1, col=1)

fig3.add_vline(x=85, line_dash='dash', line_color='#27AE60',
               annotation_text='Cible 85%', row=1, col=1)

# CMJ vs Capacité max
fig3.add_trace(go.Bar(
    name='CMJ (moyen journalier)',
    y=df_up['nom'],
    x=df_up['cmj'],
    orientation='h',
    marker_color='#4A90D9',
), row=1, col=2)

fig3.add_trace(go.Bar(
    name='Capacité max/j',
    y=df_up['nom'],
    x=df_up['capacite_max_j'],
    orientation='h',
    marker_color='#BDC3C7',
    opacity=0.6,
), row=1, col=2)

fig3.update_layout(
    height=450,
    title_text='📊 Performance des Unités de Production',
    barmode='overlay',
    paper_bgcolor='#F8F9FA',
    plot_bgcolor='#F8F9FA',
)
fig3.show()

## 8. 🗂️ Vue par site — Graphe filtré

In [8]:
def view_by_site(site_code):
    """Affiche uniquement les nœuds appartenant à un site donné."""
    nodes = [
        n for n in G.nodes
        if G.nodes[n].get('site') == site_code
    ]
    # Inclure aussi les voisins directs (flux entrants/sortants)
    neighbors = set()
    for n in nodes:
        neighbors |= set(G.predecessors(n)) | set(G.successors(n))
    all_nodes = list(set(nodes) | neighbors)

    subG = G.subgraph(all_nodes)
    site_label = site_name.get(site_code, site_code)
    fig = build_plotly_graph(subG, title=f'🏭 Vue site : {site_label} ({site_code})')
    fig.show()

# ── Choisissez le site à visualiser ──────────────────────────────────────────
# Codes disponibles :
print('Sites disponibles :')
for s in data_sites:
    print(f"  {s['code']}  →  {s['nom']} ({s['localisation']})")

view_by_site('SITE-A')   # ← modifier ici

Sites disponibles :
  SITE-A  →  Site Alpha (Lyon)
  SITE-B  →  Site Beta (Grenoble)
  SITE-C  →  Site Gamma (Marseille)
  SITE-D  →  Site Delta (Paris)


## 9. 📈 Stocks matières premières — Niveaux par rapport aux seuils

In [9]:
df_mp_ext = df_mp.copy()
df_mp_ext['pct_stock'] = (df_mp_ext['volume_stock'] / df_mp_ext['stock_max'] * 100).round(1)
df_mp_ext['statut'] = df_mp_ext.apply(
    lambda r: '🔴 Critique' if r['volume_stock'] <= r['stock_min']
              else '🟡 Bas' if r['volume_stock'] <= r['stock_min'] * 1.5
              else '🟢 OK', axis=1
)

fig4 = go.Figure()

fig4.add_trace(go.Bar(
    name='Stock actuel',
    x=df_mp_ext['nom'],
    y=df_mp_ext['volume_stock'],
    marker_color=['#E74C3C' if s.startswith('🔴') else
                  '#F39C12' if s.startswith('🟡') else '#4A90D9'
                  for s in df_mp_ext['statut']],
    text=df_mp_ext['statut'],
    textposition='outside',
))

fig4.add_trace(go.Scatter(
    name='Stock min',
    x=df_mp_ext['nom'],
    y=df_mp_ext['stock_min'],
    mode='markers+lines',
    line=dict(color='#E74C3C', dash='dash'),
    marker=dict(size=8),
))

fig4.add_trace(go.Scatter(
    name='Stock max',
    x=df_mp_ext['nom'],
    y=df_mp_ext['stock_max'],
    mode='markers+lines',
    line=dict(color='#27AE60', dash='dot'),
    marker=dict(size=8),
))

fig4.update_layout(
    title='📦 Niveaux de stock — Matières premières',
    xaxis_title='Matière première',
    yaxis_title='Volume',
    height=420,
    paper_bgcolor='#F8F9FA',
    plot_bgcolor='#F8F9FA',
)
fig4.show()

print('\nRécapitulatif stocks :')
display(df_mp_ext[['code','nom','volume_stock','stock_min','stock_max','pct_stock','statut']])


Récapitulatif stocks :


,code,nom,volume_stock,stock_min,stock_max,pct_stock,statut
0,MP-001,Résine PET,50000,10000,80000,62.5,🟢 OK
1,MP-002,Colorant Bleu,2000,500,5000,40.0,🟢 OK
2,MP-003,Acier galvanisé,120000,30000,200000,60.0,🟢 OK
3,MP-004,Huile de coupe,8000,1000,15000,53.3,🟢 OK
4,MP-005,Emballage carton,15000,3000,25000,60.0,🟢 OK


## 10. 📋 Analyse du chemin critique (flux le plus long)


In [10]:
# Chemin(s) le(s) plus long(s) dans le graphe
try:
    longest = nx.dag_longest_path(G)
    print('📍 Chemin critique identifié :')
    for step in longest:
        attr = G.nodes[step]
        site = site_name.get(attr.get('site',''), '')
        label = attr.get('label', step)
        print(f'   → [{step}] {label}  ({site})')

    print(f'\nLongueur du chemin critique : {len(longest)} étapes')

    # Nœuds goulots d'étranglement : OEE < 80%
    goulots = [
        (n, G.nodes[n].get('oee'), G.nodes[n].get('label',n))
        for n in G.nodes
        if G.nodes[n].get('oee') and G.nodes[n].get('oee') < 80
    ]
    if goulots:
        print('\n⚠️  Goulots d'étranglement (OEE < 80%) :')
        for code, oee, label in sorted(goulots, key=lambda x: x[1]):
            print(f'   🔴 {code} — {label} : OEE = {oee}%')
    else:
        print('\n✅ Aucun goulot détecté (OEE ≥ 80% partout)')

except nx.NetworkXUnfeasible:
    print('⚠️  Le graphe contient un cycle — vérifier les données.')

SyntaxError: unterminated string literal (detected at line 20) (1085412090.py, line 20)

---
## 💡 Pour enrichir ce notebook

| Action | Comment faire |
|---|---|
| Ajouter un site | Éditer `sites.json`, ajouter un objet dans `sites[]` |
| Ajouter une MP | Éditer `matieres_premieres.json` |
| Ajouter une unité | Éditer `unites_production.json`, renseigner `produits_suivants` pour connecter au graphe |
| Changer le layout | `build_plotly_graph(..., layout_algo='kamada_kawai')` |
| Filtrer un site | `view_by_site('SITE-X')` |
| Exporter le graphe | `nx.write_graphml(G, 'flux.graphml')` |